## Stability Analysis

The stability analysis evaluates whether a candidate clustering configuration produces consistent cell assignments across different random seeds.

The clustering parameters are kept fixed while the random seed is varied across repeated runs.

### First agent run/ Candidate 1

`n_neighbors=30, resolution=0.9, n_pcs=25`

- 9 clusters were recovered in all 10 runs.
- Mean pairwise ARI: `0.9126`
- Median ARI: `0.9134`
- SD: `0.0283`
- ARI range: `0.8479–0.9777`

This configuration showed high consistency across the tested random seeds.

### Second agent run/Candidate 2

`n_neighbors=30, resolution=0.7, n_pcs=30`

- 8 clusters were recovered in 8/10 runs.
- 7 clusters were recovered in 2/10 runs.
- Mean pairwise ARI: `0.9114`
- Median ARI: `0.9668`
- SD: `0.1454`
- ARI range: `0.6193–1.0000`

Although this configuration achieved substantially **better internal clustering metrics**, the stability analysis showed **greater variation across random seeds**.

This approach is consistent with how stability has been evaluated in prior scRNA-seq clustering work: repeated clustering runs and comparison of resulting partitions using measures such as ARI are established approaches. SC3, for example, evaluated clustering stability across repeated runs and also used consensus information to improve robustness. More recent work such as scICE also specifically examines variation in cluster assignments across random seeds.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)


True

In [3]:
from src import data_load as dl
from src import preprocessing_pipeline as prp
from src import preprocessing_pipeline as prp
from src import clustering as cl
from src.stability import run_stability_analysis, calculate_pairwise_ari, calculate_cluster_jaccard
import numpy as np
from collections import defaultdict

In [4]:
pbmc_data = dl.load_pbmc3k()
pbmc_data = prp.run_preprocessing_pipeline(pbmc_data)

Running QC...
Running normalization...
Running PCA...
Finished preprocessing.


In [5]:
stability_result = run_stability_analysis(
    pbmc_data,
    n_neighbors=30,
    resolution=0.7,
    n_pcs=30,
    seeds=range(10),)

print("Cluster counts:", stability_result["cluster_counts"])

Cluster counts: [8, 8, 7, 8, 8, 8, 8, 8, 8, 7]


candidate (30, 0.9, 25) appears highly consistent across the tested random seeds.

In [6]:
from collections import Counter

cluster_count_frequency = Counter(stability_result["cluster_counts"])

print("Cluster count frequency:")
for k, count in sorted(cluster_count_frequency.items()):
    print(f"K={k}: {count}/10 runs")

Cluster count frequency:
K=7: 2/10 runs
K=8: 8/10 runs


In [7]:
ari_scores = calculate_pairwise_ari(stability_result["assignments"])

print("Number of pairwise comparisons:", len(ari_scores))
print("Mean ARI:", sum(ari_scores) / len(ari_scores), end="\n")
print("-----------------")
print("Stability summary")
print("-----------------")
print(f"Mean ARI:   {np.mean(ari_scores):.4f}",  end='\n')
print(f"Median ARI: {np.median(ari_scores):.4f}",  end='\n')
print(f"SD ARI:     {np.std(ari_scores):.4f}",  end='\n')
print(f"Min ARI:    {np.min(ari_scores):.4f}",  end='\n')
print(f"Max ARI:    {np.max(ari_scores):.4f}")

Number of pairwise comparisons: 45
Mean ARI: 0.9113751226351089
-----------------
Stability summary
-----------------
Mean ARI:   0.9114
Median ARI: 0.9668
SD ARI:     0.1454
Min ARI:    0.6193
Max ARI:    1.0000
